<a href="https://colab.research.google.com/github/Subroy1/MSAI_AllPracticeModules/blob/main/Module%2011/PracticalAssignement2_used_cars_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# What drives the price of a car?

![](images/kurt.jpeg)

**OVERVIEW**

In this application, you will explore a dataset from Kaggle. The original dataset contained information on 3 million used cars. The provided dataset contains information on 426K cars to ensure speed of processing.  Your goal is to understand what factors make a car more or less expensive.  As a result of your analysis, you should provide clear recommendations to your client -- a used car dealership -- as to what consumers value in a used car.

### CRISP-DM Framework

<center>
    <img src = images/crisp.png width = 50%/>
</center>


To frame the task, throughout our practical applications, we will refer back to a standard process in industry for data projects called CRISP-DM.  This process provides a framework for working through a data problem.  Your first step in this application will be to read through a brief overview of CRISP-DM [here](https://mo-pcco.s3.us-east-1.amazonaws.com/BH-PCMLAI/module_11/readings_starter.zip).  After reading the overview, answer the questions below.

### Business Understanding

From a business perspective, we are tasked with identifying key drivers for used car prices.  In the CRISP-DM overview, we are asked to convert this business framing to a data problem definition.  Using a few sentences, reframe the task as a data task with the appropriate technical vocabulary.

**Understand the Business objective**

Worldwide, used car sales is a major business occupuation due to its ability to provide affordable transportation to millions of citizens.

Though there is no one-size-fits-all solution as location , region , country , avaiability of public mode of transportation , purpose of owning a car etc and affordability based on demographics also play a part, this exercise makes an effort to understand the fundamental principles of Machine learning algorithm,, specifically LR techniques (Linear and Multiple Linear , Ridge ,Lasso and Elastic Net regression models) in order to price cars based on available features and derived features with additional application of domain knowledge in this field.

Furthermore, since the size of the dataset is huge (400K plus), in order to avoid the model from memorizing the relationships between independent and target variable (price), L1 and L2 regularization and subsequent hyperparamter tuning have been relied upon for creating the optimum model.

During the course of implementing this data mining problem , we would have to constantly refine our solution by not only going back to refine data as needed and repeated remodelling but also by enriching our business understanding from evaluation step.

Finally a summart report would be provided to the intended audience namely used car dealers or franchises who would restructure their inventory accordingly to boost sales.

In [3]:
# All of the imports needed for the exercise consolidated in the beginning .
#Standard imports
import pandas as pd
import numpy as np

#scikit-learn libs
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler , TargetEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Plotting libs
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

#ignore warnings
import warnings
warnings.filterwarnings('ignore')

**Loading the Data**



In [9]:
df = pd.read_csv("/content/sample_data/vehicles.csv")
df.head()

,id,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,state
0,7222695916,prescott,6000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,az
1,7218891961,fayetteville,11900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ar
2,7221797935,florida keys,21000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,fl
3,7222270760,worcester / central MA,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ma
4,7210384030,greensboro,4900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,nc


# Data understanding the features

*  **price - Our target variable**
*  manufacturer - company manufacturing the brand
*  model - model of the specific make of the car
*  year- year of manufacturing the car or bought by 1st owner
*  transmission - whether manual or automatic gear  
*  drive - whether all wheel drive or 2 wheel drive (Front or rear)
*  state - place where car is registered/ manufactured
*  region- region within the state
*  condition - good or bad (ordinal feature , might need encoding )
*  cylinders- specific to the combustion engine , applicable for petrol and diesel cars only
*   size- such as small , medium , SUV- large , coupe , sedan -medium like that *(might need encoding )
*   fuel- whether runs on gasoline or  diesel , electric (encoding needed)
*   title_status - in which form is the vehicle , it is clean , or rebuilt etc .
* paint_color - car color hardly plays any role in prediction
* odometer- the mileage of the car, how many miles / kms it has driven
* id , VIN - identifiers which do not play any role in predictive algorithms (should be dropped for model building)
* type- utility of the vehicle such as pick up or the structure such as sedan or coupe etc

In [24]:
# random inspection as initial few records were NaN for majority of columns in the dataset
df.iloc[150000:150002]


,region,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,drive,size,type,paint_color,state
150000,bloomington,38590,2019.0,ram,1500 crew cab big horn,good,6 cylinders,gas,28556.0,clean,other,4wd,NaN,pickup,white,in
150001,bloomington,43990,2019.0,ram,1500 crew cab big horn,good,8 cylinders,gas,17352.0,clean,other,4wd,NaN,pickup,black,in


In [23]:
#We will take the apporach of removing the feature completely whereby they have no role to play # in pricing vehicles and loads of them are null .
# for eg - VIN is a vehicle identification number and has nothing to do with car pricing .
df=df.drop(columns={"VIN","id"})

In [25]:
df['title_status'].unique()

array([nan, 'clean', 'rebuilt', 'lien', 'salvage', 'missing',
       'parts only'], dtype=object)

**Data Preprocessing**

Data Cleaning - Broadly, we will follow the following principle .
1. Any cleaning based on domain knowledge , identifying outliers would be done pre train_test_split
2. Any data preprocessing including data cleaning based on statistical properties would be done after train test split to avoid data leakage.





In [77]:
# Inpsect null values in the dataset , aim to identify null features adding 0 value for prediction such as identifiers .
#We observe that many features have loads of null values,
df.isnull().sum().sort_values(ascending=False)

,0
size,306293
cylinders,177610
condition,174036
drive,130499
paint_color,130135
type,92790
manufacturer,17578
title_status,8174
model,5209
odometer,4332


In [70]:
# Unique models are

#df['model'].value_counts().sort_values(ascending=False).to_csv('/content/sample_data/model.csv', index=False)
len(df['model'].value_counts())#29649 models
#we need to reduce cardinality , after inspecting the csv above, I decide to keep only this models whose value counts >0.5% of total 426K(~ 2100), group all remaining to 'others' category.
#This would ensure model stablity  by reducing sparsity.

#model_counts= df['model'].value_counts()
#model_counts_df= pd.DataFrame(model_counts)


#vehicles_model_df = pd.DataFrame(df['model'].value_counts())
# cap_price=np.quantile(df['price'],0.95)
# df[df['price'] >cap_price].shape

#top_models=np.quantile(vehicles_model_df['count'], 0.95)
#df[df['model']!='f-150'].tail(10)
# encode the missing size to 0





In [43]:
# Unique states and regions
len(df['state'].unique())#  51 us states - keep
len(df['region'].unique())#  404 us states

404

# **Exploratory Data Analysis**

1.   List item
2.   List item



*Describe the different features within the data*

*Data preprocessing*



**Following CRISP -DM model**
We would first understand the data, describe the statistical properties , observe the categorical and numerical columns (features), define the features for understability .

We will also inspect the quality of the data , null values and any transformations needed.



In [ ]:
df.describe().apply(lambda s: s.apply('{0:.5f}'.format))# Print numerical values , can be seen that price has maximum value seemingly as outliers due to sheer absurdness of the value ,
# Since we do not know how many of such values are obviously wrong , we would print the 95 and 99%ile for inspecting further .


,id,price,year,odometer
count,426880.00000,426880.00000,425675.00000,422480.00000
mean,7311486634.22433,75199.03319,2011.23519,98043.33144
std,4473170.41256,12182282.17360,9.45212,213881.50080
min,7207408119.00000,0.00000,1900.00000,0.00000
25%,7308143339.25000,5900.00000,2008.00000,37704.00000
50%,7312620821.00000,13950.00000,2013.00000,85548.00000
75%,7315253543.50000,26485.75000,2017.00000,133542.50000
max,7317101084.00000,3736928711.00000,2022.00000,10000000.00000


In [ ]:
cap_price=np.quantile(df['price'],0.95)
df[df['price'] >cap_price].shape



(21311, 18)

In [ ]:
# Data preprocessing and cleaning
# Inspect empty values
# First remove features which are blank and count > 70% of the size of the dataset
# Data cleaning- Remove too old data as they were in a different generation
# Remove outliers which are aberrations and belong to 5% beyond range of numericals (beyond 0.95 quantile) - PLot the price or target price , log(price) to show normal distribution
# Plot the corelation matrix and heatmap, plot the linear relationship supposedly between different features and target variable.
np.percentile(df['price'], 99)
# Explore categorical data and create features using encoders.
#target_names=

In [ ]:
#df.tail(2)
print(df['size'].unique())
print(df['drive'].unique())


In [ ]:
df['log_price'] = np.log10(df['price'])

In [ ]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)
print(df.describe())
pd.reset_option('display.float_format')

In [ ]:
nitems =  len(df['price'])
df['price'].sort_values(ascending=False).iloc[:20]

In [ ]:
# clearly , the max price is an abberation and hence an outlier to be filtered out (3736928711.000) .
import plotly.express as px
px.histogram(x=df['price'], nbins=5)

### Data Understanding

After considering the business understanding, we want to get familiar with our data.  Write down some steps that you would take to get to know the dataset and identify any quality issues within.  Take time to get to know the dataset and explore what information it contains and how this could be used to inform your business understanding.

### Data Preparation

After our initial exploration and fine-tuning of the business understanding, it is time to construct our final dataset prior to modeling.  Here, we want to make sure to handle any integrity issues and cleaning, the engineering of new features, any transformations that we believe should happen (scaling, logarithms, normalization, etc.), and general preparation for modeling with `sklearn`.

In [ ]:
# REAL DATA CLEANING , ALL ARE NULL -  no statistical loss to model because mandatory features missing from dataset
df = df[~ (df['model'].isnull() & df['year'].isnull()  &  df['condition'].isnull() & df['year'].isnull() & df['fuel'].isnull() & df['odometer'].isnull() & df['title_status'].isnull() & \
   df['transmission'].isnull() & df['drive'].isnull() & df['size'].isnull() & df['type'].isnull() & df['paint_color'].isnull() & df['manufacturer'].isnull()  & df['cylinders'].isnull())]

### Modeling

With your (almost?) final dataset in hand, it is now time to build some models.  Here, you should build a number of different regression models with the price as the target.  In building your models, you should explore different parameters and be sure to cross-validate your findings.

### Evaluation

With some modeling accomplished, we aim to reflect on what we identify as a high-quality model and what we are able to learn from this.  We should review our business objective and explore how well we can provide meaningful insight into drivers of used car prices.  Your goal now is to distill your findings and determine whether the earlier phases need revisitation and adjustment or if you have information of value to bring back to your client.

### Deployment

Now that we've settled on our models and findings, it is time to deliver the information to the client.  You should organize your work as a basic report that details your primary findings.  Keep in mind that your audience is a group of used car dealers interested in fine-tuning their inventory.

In [ ]:
# What features make a car more or less expensive .
# Provide clear recommedations to used car dealers about what customers value in a used car
#After understanding, preparing, and modeling your data, write up a basic report that details your primary findings. Your audience for this report is a group of used car dealers interested in fine-tuning their inventory.
# From existing experience , you can tailor your findings towards manufacturer's dealerships and private individual car dealers distinctively .

**Next steps with recommendations**

**Luxury tax** Some features of a car are region specific , for eg - In the UK, we are required to pay luxury tax on top of regular tax for each year for vehicles having RRP > 40K £ . Hence , a feature like “IsLuxTaxPayable” can be a critical determinant for some purchasers especially when even for the same car brand and model and same manufacturing year, a submodel can edge past the RRP beyond 40K threshold . This is called Luxury Trap here in the UK and adds to running costs of vehicles.
Other features of interest and in normal practice

1.**Front wheel drive** - Geography is important here as in hilly areas (such as Scotland which is also in UK), stability  of cars is more important hence 2.
**All-wheel-drive** (instead of FWD ) is more desirable to customers and often shoots up prices.

2.**Running Costs**- Insurance group taken into consideration for pricing vehicles because luxury cars often have higher premiums. Annual Servicing charges also are a factor with preium cars.

3.**Accident history of cars**(AccidentCategory -  S/N/NaN) - Whether the car had an accident category , whether the damage was structural or non-structural. significantly lowers the price relatively .

4.**Manufacturer approved** - Whether car dealer belongs to official manufacturers  dealerships such as BMW , Porsche or private dealers .

**Limitations**

**Generalization to new makes** When new car brands and models come out , there wouldnt be enough data to make accurate predictions as there would be many unknowns in the existing features.

**Time based sales** Car sales is also heavily impacted by seasonal variations including festivals and occassions such as year end when demand is high due to holiday season, hence the algo has to evolve to consider time series data for dealers to price vehicles in order to make more profits .

**Manufacturing warranty**- Customers increasingly value existing warranty offered by original manufacturer which normally lasts for 3 years and hence in spite of a general assumption that car depreciates heavily in the first 3 years , this is an important factor to consider as desirability of customers would be high for buying such cars.

**Consumer preference** The above dataset does not have any indicator for consumer preference hence additional segmentation data is required for brand conscious customers (and could afford) who would buy only elite brands vs medium-tier customers hence in real-world sophisticated models are needed tailored for different brands.

**Checklist for a proer submission**

Built baseline Linear Regression

Showed coefficient interpretation

Added polynomial terms

Compared Ridge vs Lasso

Did cross-validation properly

Used SFS or RFE

Explained bias–variance tradeoff

Discussed overfitting

And most importantly:

Explained results in business language.